In [ ]:
# !pip install trl bitsandbytes

In [ ]:
"""
Fine-tune Qwen2.5-7B-Instruct into a sarcastic/satirical chatbot using QLoRA.

Hardware budget this is tuned for:
  - GPU: ~3.5-4 GB VRAM peak (4-bit base model + LoRA + batch size 1 + grad checkpointing)
  - RAM: ~3-4 GB (small dataset loaded fully in memory)

Install (do this once, in a venv):
  pip install torch --index-url https://download.pytorch.org/whl/cu121
  pip install transformers peft trl bitsandbytes datasets accelerate
"""

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import warnings
warnings.filterwarnings('ignore')

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
DATA_PATH = "./drive/MyDrive/SaasLM/sarcasm_dataset.jsonl"   # your instruction pairs (see format below)
OUTPUT_DIR = "./drive/MyDrive/SaasLM/sarcastic-qwen-lora v1.0"

In [ ]:
# ----------------------------------------------------------------------
# 1. Dataset
# ----------------------------------------------------------------------
# Expected JSONL format, one object per line:
# {"messages": [
#    {"role": "user", "content": "How do I boil an egg?"},
#    {"role": "assistant", "content": "Ah yes, humanity's greatest unsolved mystery. Put egg in hot water. Wait. Somehow, civilization survives another day."}
# ]}
#
# Aim for 500-2000 examples. Quality and consistent voice matter far more
# than quantity for a style-transfer task like this.

dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset = dataset.train_test_split(test_size=0.05, seed=42)

In [ ]:
# ----------------------------------------------------------------------
# 2. Load base model in 4-bit (QLoRA)
# ----------------------------------------------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # use torch.float16 if your GPU lacks bf16
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

In [ ]:
# ----------------------------------------------------------------------
# 3. LoRA config — small rank keeps VRAM and trainable params tiny
# ----------------------------------------------------------------------
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)